In [ ]:
# Example 1: Basic formula parsing - build a design matrix from a pandas DataFrame.
import pandas as pd
from formulaic import Formula

df = pd.DataFrame({
    'y': [1.0, 2.0, 3.0, 4.0, 5.0],
    'x': [1.0, 2.0, 3.0, 4.0, 5.0],
    'z': [0.5, 1.5, 2.5, 3.5, 4.5],
})

f = Formula('y ~ x + z')
print(f"Formula: {f}")

y, X = f.get_model_matrix(df)
print(f"y shape: {y.shape}")
print(f"X shape: {X.shape}")
print(f"X columns: {list(X.columns)}")

assert 'Intercept' in X.columns
assert 'x' in X.columns
assert 'z' in X.columns
assert X.shape == (5, 3)


In [ ]:
# Example 2: Categorical variables expanded via one-hot encoding.
import pandas as pd
from formulaic import Formula

df = pd.DataFrame({
    'y': [1.0, 2.0, 3.0, 4.0, 5.0, 6.0],
    'group': ['a', 'b', 'c', 'a', 'b', 'c'],
})

f = Formula('y ~ C(group)')
y, X = f.get_model_matrix(df)

print(f"X columns: {list(X.columns)}")
print(X)

# Reference coding drops one level (treatment contrast default).
assert X.shape[0] == 6
assert X.shape[1] == 3  # Intercept + 2 dummies for 3-level factor


In [ ]:
# Example 3: Interaction terms and transformations using inline Python.
import numpy as np
import pandas as pd
from formulaic import Formula

rng = np.random.default_rng(42)
df = pd.DataFrame({
    'y': rng.normal(size=20),
    'a': rng.normal(size=20),
    'b': rng.normal(size=20),
})

f = Formula('y ~ a + b + a:b + np.log(np.abs(a) + 1)')
y, X = f.get_model_matrix(df)

print(f"X columns: {list(X.columns)}")
assert 'a:b' in X.columns
assert any('log' in c for c in X.columns)
assert X.shape == (20, 5)


In [ ]:
# Example 4: ModelSpec round-trip — apply a fit formula to new data.
import pandas as pd
from formulaic import Formula

train = pd.DataFrame({
    'y': [1.0, 2.0, 3.0, 4.0],
    'group': ['a', 'b', 'a', 'b'],
    'x': [10.0, 20.0, 30.0, 40.0],
})

y_train, X_train = Formula('y ~ C(group) + x').get_model_matrix(train)
model_spec = X_train.model_spec

test = pd.DataFrame({'group': ['a', 'b'], 'x': [50.0, 60.0]})
X_test = model_spec.get_model_matrix(test)

print(f"Train columns: {list(X_train.columns)}")
print(f"Test columns:  {list(X_test.columns)}")

assert list(X_train.columns) == list(X_test.columns)
assert X_test.shape == (2, 3)


In [ ]:
# Example 5: Sparse output materialization for large categorical designs.
import pandas as pd
import scipy.sparse as sp
from formulaic import Formula

df = pd.DataFrame({
    'y': [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0],
    'cat': ['a', 'b', 'c', 'd', 'a', 'b', 'c', 'd'],
    'x': [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0],
})

f = Formula('y ~ C(cat) + x')
y, X = f.get_model_matrix(df, output='sparse')

print(f"X type: {type(X).__name__}")
print(f"X shape: {X.shape}")
print(f"X nnz:   {X.nnz}")

assert sp.issparse(X)
assert X.shape == (8, 5)  # Intercept + 3 dummies + x
